# 🗣️ Cloning 챕터 종합 정리 노트 (4주차 — 보이스 클로닝)

> **생성형 AI 기반 음성 에이전트 개발 과정 · 보이스 클로닝 파트 리뷰**
> `Cloning/` 폴더의 실습 노트북 6개를 하나로 통합한 **복습·재사용용 노트**입니다.

| 항목 | 내용 |
|---|---|
| 대상 노트북 | `Cloning/` 6개 (4-V 개념 ~ 4-Q Qwen3-TTS) |
| 노트의 목적 | ① **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | Google Colab(T4) 실습 기준 / **macOS(Apple Silicon) 실행 가이드 포함** |
| 과정 공통 표준 | 동의 계약 · TTS 계약 상속 · SECS+CER **이중 저울** · 3초 참조 · 게이트로 권리 지키기 |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **GPU·모델·네트워크 없이 실행되는** 계약·저울·등록부·수식·어댑터만 모았습니다.
> 실제 엔진(Qwen3-TTS, ElevenLabs API, resemblyzer) 호출이 필요한 부분은 시그니처+설명으로 요약했습니다 (2장에서 ✅/📄로 구분). 위에서 아래로 실행하세요.


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 (6개 중요도·의존 관계·macOS 지원·카드 구조 해설) |
| **1** | 실험에 필요한 선행 지식 (7개 주제) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (노트북별 가이드 + macOS 실습법 + 판단 기준) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 6개 노트북 한눈에

| 노트북 | 주제 | 중요도 | macOS 실행 | 코멘트 |
|---|---|---|---|---|
| **4-V** | 보이스 클론 개념 — "3초의 무게" | ★★★ 입문 | ⚠️ 일부 | 동의 계약·SECS 저울·클로닝 카드 1호 |
| **4-E** | 화자 임베딩 추출 — 저울의 해부학 | ★★★ 입문 | ✅ | 임베딩 계약·1.6초 창·3초의 무게·임베딩 카드 |
| **4-M** | 다중 화자 클로닝 비교 | ★★★ **방법론 핵심** | ⚠️ 일부 | 화자 등록부·교차 행렬·intra/inter 임계값·전사 오염 |
| **4-O** | 클로닝 모델 최적화 | ★★ 최적화 | ⚠️ 일부 | 병목 해부·레버 A~E·파레토·운영점 계약화 |
| **4-Q** | Qwen3-TTS 기반 클로닝 | ★★ 최적화·정리 | ⚠️ 일부 | 사전검증·동의 게이트·ICL vs x-vector·A/B 삼종 |
| **4-EL** | ElevenLabs 감정 클로닝 (API) | ★ API | ✅ (키 필요) | 동의 계약 v2(제3자 반출)·감정 태그·원가 계량기 |

> **클로닝 2대 기전**: **임베딩형**(4-V의 `_clone` — x-vector를 프롬프트로) vs **프롬프트 연속형**(4-Q의 `generate_voice_clone` — 참조 오디오+전사를 프롬프트에). 입력 계약(전사 필요 여부)이 갈림길입니다.

## 0-2. 학습 흐름 (의존 관계)

```
4-V(개념·동의·SECS) ──→ 4-E(화자 임베딩·저울 해부학) ──→ 4-M(다중 화자 비교·등록부)
   │                                                          │
   │             4-EL(외부 API·감정·원가) ── 병렬 ────────────┤
   │                                                          ↓
   └──────────────────────────────→ 4-Q(클로닝·최적화 A/B) ←── 4-O(레버·파레토)
```

- **4-V → 4-E**: 저울(SECS)이 왜 움직이는지, 그 기전을 임베딩에서 분해한다.
- **4-M → 4-Q**: 같은 모델(Qwen3-TTS)로 등록부 비교(4-M) → 최적화(4-O) → 정리(4-Q). **3연작은 같은 방법론 반복**이라 이 노트에서 통합합니다.
- **4-EL**: 로컬 실습에 없던 축 **원가**와 **감정 태그**를 상용 API로 실험.

## 0-3. 클로닝 4대 표준 (모든 실험의 무대)

1. **동의 계약** — `{speaker_id, source_type, scope, obtained_at, notice_given, expires_at}`.
   `source_type` 3종(실음성 동의/합성기계/자기 음성) · `scope` 3단계(좁은→넓은: education_lab → internal_poc → production).
   **클로닝은 프롬프트가 아니라 게이트로 통과시킨다**(`clone_guard`).
2. **이중 저울** — 목소리는 **SECS**(화자 임베딩 코사인), 내용은 **CER**(왕복 전사). 한 축만 보면 속는다.
3. **3초의 무게** — 참조 길이 3초만으로 벡터가 충분히 안정된다(4-E 실측). 그 '충분함'이 곧 위험의 크기.
4. **양쪽 문** — 입력측: 동의 계약 게이트. 출력측: 워터마크 서명. 둘 중 하나만으로는 반쪽.

## 0-4. 카드 요약

| 카드 | 출처 | 핵심 |
|---|---|---|
| **임베딩 카드** | 4-E | 계약(차원·L2=1·float32) · 1.6초 창 · 일찍 안정 · 5dB에서 크게 밀림 |
| **클로닝 카드 1호** | 4-V | 기전 2종 · 이중 저울 · 3초 실험 · 동의 게이트 · 양쪽 문 |


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전 훑기 → ② 1장 선행 지식 → ③ 2장 함수 실행하며
> "검증 통과 ✅" 눈으로 확인. 모르는 단어는 여기로 돌아오세요.

### A. 클로닝 개념
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 보이스 클로닝 | 특정 화자의 목소리로 새 문장을 합성 | "3초 녹음으로 목소리 복제" |
| 임베딩형 클로닝 | 화자 목소리 특징을 벡터로 추출해 참조로 사용 | x-vector 계열 (4-E) |
| 프롬프트 연속형 | 참조 오디오+전사를 프롬프트에 넣어 생성 | Qwen3-TTS·ElevenLabs 계열 (4-Q/4-EL) |
| 화자 임베딩 | 목소리 특징을 담은 숫자 벡터 (보통 단위벡터) | "목소리 지문" — SECS 저울의 재료 |
| TTS 계약 | 음성 레코드의 표준 형식 (16kHz mono float32) | ASR/TTS와 공유하는 시스템 표준 |

### B. 동의와 권리 — "목소리는 자산이다"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 동의 계약 (consent) | 누구·무엇·어디까지·언제까지 쓰는가의 기록 | 클로닝 이전 필수 관문 |
| scope (범위) | education_lab < internal_poc < production | 좁은 범위는 넓은 범위를 대체 못 함 |
| 클론 게이트 | 동의 통과 시에만 클로닝 실행 | PermissionError로 강제 |
| 워터마크 | 합성음을 구분하는 표시 | 사후 검증 장치 |
| 라이선스 | 모델·목소리의 상업 사용 조건 | 배포 전 반드시 확인 |

### C. 평가 저울 — "내용과 목소리는 다른 저울"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| CER | 글자 오류율 — 내용이 맞는가 | 3-ST 왕복 심판 |
| SECS | 화자 임베딩 코사인 유사도 — 목소리가 같은가 | **절대 눈금 아님** → 기준선 대비 |
| 3점 캘리브레이션 | 자기>같은 화자>다른 화자 순서로 기준선 세우기 | SECS 해석의 전제 |
| A/B 삼종 | 파형 정합: dur_ratio·rms_db_diff·corr | 오디오가 "같은 신호"인가 |
| 임베딩 계약 | {dim, norm:1.0, dtype:float32} | 화자 벡터의 표준 |

### D. 다중 화자·지연·최적화
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 화자 등록부 (registry) | 화자별 임베딩을 모아 두는 DB | 여러 화자 비교의 출발점 |
| intra/inter | 같은 화자 내 / 다른 화자 간 유사도 | 임계값 설계의 두 극단 |
| 잘림 (truncation) | 긴 문장이 중간에 끊김 | 토큰 상한의 원인 |
| max_new_tokens | 생성 허용 최대 토큰 수 | 지연·원가와 직결 |
| probe | 소스 코드를 읽어 구조 파악 | "추측 말고 소스를 보라" |
| 파레토 (pareto) | 어느 쪽도 나아지지 않는 절충점들 | 최적화의 후보 집합 |
| 운영점 계약화 | 파레토에서 고른 점을 INFER_CONTRACT로 고정 | "고른 설정을 계약으로" |

### E. 배경 지식 — 이 챕터가 왜 존재하는가
TTS가 "말하는 입"이라면, **클로닝은 "특정 사람의 입"을 만드는 것**입니다. 그래서 이 노트의 중심은
기술(합성)보다 **권리(동의)**와 **검증(이중 저울)**입니다.
1. **기술의 앞에 게이트**: 목소리는 돈과 신뢰의 문제 → 합성 함수를 동의 게이트로 감싼다.
2. **두 가지를 따로 검증**: 내용이 맞아도 목소리가 다르면 실패, 목소리가 같아도 내용이 틀리면 실패 —
   CER(내용)과 SECS(목소리)를 **이중 저울**로 재야 한다.
3. **상대 눈금의 함정**: SECS 0.9가 "좋다"가 아니라 **3점 기준선**(자기>같은>다른) 대비로 판독한다.


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 핵심 개념 |
|---|---|---|---|
| 2.0 | `make_consent_record`·`validate_consent`·`clone_guard` | 동의 계약 v1 + 클론 게이트 | 권리 → 기술 앞 |
| 2.1 | `cer`·`secs`·`_MockEncoder` | 이중 저울 (내용·목소리) | CER + SECS + 3점 캘리브레이션 |
| 2.2 | A/B 삼종·`validate_emb` | 파형 정합 + 임베딩 계약 | 같은 신호? 같은 목소리? |
| 2.3 | `SpeakerRegistry`·`audit_registry` | 화자 등록 + 감사 | 등록 조건·cps 상식 |
| 2.4 | `mnt_for`·`est_max_new_tokens`·`predicted_steps` | 토큰 상한 수식 | 지연·원가 예측 |
| 2.5 | `probe`·`is_pareto`·`synth_optimized` | 최적화 도구 | 구조 읽기 → 파레토 → 계약 |
| 2.6 | ElevenLabs 어댑터 | 감정 클로닝 (API 미호출) | 동의 v2·감정 태그·원가 계량기 |


# 1. 실험에 필요한 선행 지식 🧠

## 1-1. 보이스 클로닝이란 — TTS와 무엇이 다른가 (4-V)

```
일반 TTS:  텍스트 ──→ ① 텍스트 분석 ──→ ② 음향 모델(누구의 목소리?) ──→ ③ 보코더
보이스 클로닝: 참조 오디오(3초) ──→ 화자 임베딩(x-vector) ──→ ②로 주입 ──→ 같은 목소리로 낭독
```

| 항목 | 일반 TTS | 보이스 클로닝 |
|---|---|---|
| 목소리 결정 | 사전 고정 (엔진 선택) | **참조 오디오가 결정** |
| 최소 자산 | 없음 | **참조 3초 + 동의** |
| 위험 | 읽기 오류 | **누구 목소리인가 (권리)** — 그래서 동의 계약이 먼저 |

- Qwen3-TTS 클로닝 모델: `Qwen/Qwen3-TTS-12Hz-0.6B-Base` — **Base 계열만 voice clone 지원** (모델명의 `12Hz` = 초당 코덱 프레임 수 `CODEC_HZ=12.0`).
- 4-Q 셀 0의 교훈: **"이름이 같다고 같은 물건이 아니다"** — 후보 패키지를 설치하기 전에 wheel 메타데이터를 받아서 비교한다.

## 1-2. 동의 계약 — 클로닝 이전에 반드시 통과해야 하는 관문 (4-V → 4-EL)

| 키 | 의미 | 값 |
|---|---|---|
| `source_type` | 목소리의 출처 | `real_voice_consented` / `synthetic_machine` / `self_voice` |
| `scope` | 사용 범위 (좁은→넓은) | `education_lab` < `internal_poc` < `production` |
| `notice_given` | 실음성 사용자에게 고지했나 | bool |
| `expires_at` | 만료 (기본 30일) | timestamp |

- **v1(4-V) → v2(4-EL) 진화**: API 실습이 새 조항을 강제했다 — `third_party_transfer`(제3자 반출)와 `processor`(처리자). *"그 목소리를 우리 서버 밖으로 내보내도 되는가"* — 외부 API에 업로드한다는 것은 곧 반출이다.
- `clone_guard(clone_fn)` — 클로닝 함수를 동의 게이트로 감싼다. 위반 시 `PermissionError`.
- 4-V의 강조: **여기서 통과해버리는 케이스가 하나라도 나오면 그 게이트는 없는 것과 같다** — 의도적 위반 후 포획이 관례.

## 1-3. 이중 저울 — 목소리와 내용은 다른 저울 (4-V · 4-M · 4-Q)

| 저울 | 측정 대상 | 방법 | 판독 |
|---|---|---|---|
| **SECS** | 목소리 유사도 | 화자 임베딩 코사인 (0~1) | ⚠️ **절대 눈금 아님** — 3점 캘리브레이션 대비 상대 판독 |
| **CER** | 내용 충실도 | 합성 → 역전사 → 오차율 | 작을수록 좋음 (3-ST 왕복 심판) |
| **A/B 삼종** | 파형 정합 | `dur_ratio`·`rms_db_diff`·`corr` | 대조군 A vs 변형 B의 차이 |

- **3점 캘리브레이션(4-V)**: ① 자기 자신(=1.0) ② 같은 화자·다른 발화 ③ 다른 화자 — 이후 모든 SECS는 **이 세 값 사이 어디인가**로 읽는다.
- `dual_scale(rec, ref_wav16)` — 두 저울을 나란히: 내용 CER × 목소리 SECS. **한 축만 보면 속는다**.
- SECS는 저울이자 재료가 같은 뿌리다: 임베딩형 클로닝의 인코더 = resemblyzer `VoiceEncoder` — **저울과 기전이 같은 부류**.

## 1-4. 화자 임베딩 — 저울의 해부학 (4-E)

- **1.6초 창**: 인코더는 `~1.6s` 부분 발화 창으로 발화를 '훑고', 부분 임베딩들의 평균이 최종 벡터. (rate ≈ 1.3창/초)
- **임베딩 계약** `EMB_CONTRACT = {dim, norm: 1.0, dtype: float32}` — 단위벡터 계약이므로 **코사인 = 내적**.
- **3초의 무게**: 벡터는 놀랄 만큼 **일찍 안정**된다 (4-E 실측 곡선). 3초가 인코더에게 창 두어 개짜리 증거라는 것.
  - ⚠️ 판독 규칙: 곡선은 **단조가 아닐 수 있다** — 1.6s 0.98인데 2.0s에서 0.93으로 일시 하락 가능 (어느 구간이 잘렸는가에 민감).
- **잡음 스윕**: 5dB에서 임베딩은 이미 크게 밀려난다 (실측 20dB 0.71 → 5dB 0.36). 4-V에서 '오염 참조 클로닝의 SECS 하락'은 엔진 실패 전에 **참조 벡터 자체의 이동**이었다 — 저울과 재료를 분리해서 읽어야 한다.


## 1-5. 다중 화자 비교 — 등록부와 임계값 (4-M)

- **화자 등록부** `SpeakerRegistry` — 화자별 동의를 강제하는 자료구조. 등록 조건: 전사 수 == 오디오 수, **발화 ≥ 2개**(intra 기준선을 만들려면).
- **교차 유사도 행렬**: 대각선(intra)과 비대각선(inter)을 함께 본다. **"대각선만 보면 속는다."**
- **분리 간극** `GAP = 화자내 최소(INTRA_MIN) − 화자간 최대(INTER_MAX)`:
  - `GAP <= 0` → 화자 구별 자체가 성립 안 함 (목소리가 비슷하거나, 녹음 조건이 화자보다 큰 변수).
  - `GAP < 0.10` → 임계값 해상도가 낮다. `GAP >= 0.10` → 클로닝 평가가 의미를 갖는다.
- **전사 게이트**: 발화 속도(자/초) 상식 검사 `2.0 ≤ cps ≤ 10.0` — 에러 없이 품질만 망가지는 **전사 교차 오염**을 잡는다.
- 4-M의 유보: 화자마다 문장이 다르면(외부 오디오+ASR 전사) 내용 차이가 섞이므로 GAP은 **보수적으로** 읽는다.

## 1-6. 지연 최적화 — 느린 이유를 소스에서 특정한다 (4-O · 4-Q)

- **병목 해부**: `probe()`로 설치된 객체의 실제 구조를 소스에서 읽는다 (문서값 32를 믿지 않는다).
- `predicted_steps(audio_sec)` = `(audio_sec × CODEC_HZ, × NUM_CODE_GROUPS)` — 생성 스텝 수를 수식으로 예측.
- **레버 5종** (4-O):

| 레버 | 무엇을 바꾸나 | 방향 |
|---|---|---|
| A. `max_new_tokens` | 스텝 상한 | 문장 길이로 계산 (과잉 할당이 느림) |
| B. `non_streaming_mode` | 텍스트를 프리필로 | A/B 실측 |
| C. **배치** | 한 번에 여러 문장 | **오늘의 핵심** — 길이 버킷팅(비슷한 것끼리) 효과 |
| D. 서브토커 샘플링 | 디코딩 전략 | 그리디 vs 샘플링 |
| E. 프롬프트 축 | 참조 길이 · 클로닝 모드 | 참조를 줄이면 얼마나 빨라지는가 |

- **파레토 프론티어**: `is_pareto` — "더 빠르고(sec 작고) 더 닮은(secs 큰) 점이 존재하면 열등." 이후 **운영점 선택** → `INFER_CONTRACT`로 계약 고정.
- 4-Q의 시간 누수 A/B: **ICL(인컨텍스트) vs x-vector** — ICL은 참조 전사(`ref_text`)가 생명줄이지만 느리고, x-vector는 전사 없이 빠르다. `est_max_new_tokens(text) = max(글자수/3.0 × CODEC_HZ × 1.5, 96)`.
- **워밍업**: 첫 호출의 커널 초기화 비용을 측정에서 제외. TTS 예산 상속(작업 가정 0.5s).

## 1-7. macOS(Apple Silicon) 실행 가이드 🍎 (추가 조사)

| 구간 | 로컬 실행 | 비고 |
|---|---|---|
| 동의 계약·등록부·CER·토큰 수식 | ✅ 가능 | 순수 Python — 이 노트 2장 전체 |
| SECS (화자 임베딩 코사인) | ✅ 가능 | `pip install resemblyzer` — CPU에서 작동 |
| 화자 임베딩 추출·SNR 스윕(4-E) | ✅ 가능 | resemblyzer + gTTS(대역 참조) |
| **Qwen3-TTS 클로닝 (4-M/4-O/4-Q)** | ⚠️ Colab/GPU 필수 | 0.6B 모델·torch — MPS로 시도는 가능하나 공식 실습은 T4 기준 |
| ElevenLabs API (4-EL) | ✅ 가능 | 키 필요, 네트워크만 있으면 macOS에서 동작 — SECS·CER 저울은 로컬에서 |

> **권장 macOS 구성**: 모델 실습(4-M/4-O/4-Q)은 Colab, **계약·저울·정합성 검사는 로컬 Jupyter**에서 — 둘 사이에 '산출물(16k wav + 메타데이터)'을 두고 통합합니다.


# 2. 실험에 필요한 함수/클래스 정의 및 주석 🔧

> **✅ 실행 코드** (GPU·모델·네트워크 없이 실행 — 자가 점검 포함) / **📄 요약** (시그니처+설명만).
> 셀 2.0→2.6 순서로 실행하세요. 2.3은 2.0의 계약 함수를, 2.5는 2.4의 수식을 사용합니다.


In [ ]:
# ═══ 2.0 동의 계약 v1 + 클론 게이트 (4-V · 4-EL) — 클로닝은 게이트로 통과시킨다 ═══
# ▶ 동의 계약은 '누구·무엇·어디까지·언제까지' 4가지를 기록한다 (출처·범위·고지·만료).
#   validate_consent는 위반 목록을 반환(진단만) — clone_guard가 이걸 게이트로 써서 PermissionError를 던진다.
#   scope는 좁은 범위가 넓은 범위를 대체할 수 없다: education_lab 승인으로 production 클로닝 불가.
import time

# v1 (4-V): 출처·범위·고지·만료 — "누구의 목소리를 쓰는가"
CONSENT_CONTRACT_KEYS = ("speaker_id", "source_type", "scope", "obtained_at",
                         "notice_given", "expires_at")
VALID_SOURCE_TYPES = ("real_voice_consented", "synthetic_machine", "self_voice")
VALID_SCOPES = ("education_lab", "internal_poc", "production")   # 좁은 → 넓은

def make_consent_record(speaker_id, source_type, scope, notice_given, valid_days=30):
    """동의 기록 생성 — 유효기간 기본 30일."""
    now = time.time()
    return {"speaker_id": speaker_id, "source_type": source_type, "scope": scope,
            "obtained_at": now, "notice_given": bool(notice_given),
            "expires_at": now + valid_days * 86400}

def validate_consent(rec, required_scope):
    """위반 목록 반환 — 진단만 한다 (빈 리스트 = 통과). TTS 계약과 같은 문법."""
    v = []
    missing = [k for k in CONSENT_CONTRACT_KEYS if k not in rec]
    if missing:
        v.append(f"누락된 키: {missing}")
        return v
    if rec["source_type"] not in VALID_SOURCE_TYPES:
        v.append(f"미허용 출처: {rec['source_type']}")
    if rec["source_type"] == "real_voice_consented" and not rec["notice_given"]:
        v.append("실음성인데 고지 미이행 — 3-ST 가중치·3-CB 워터마크가 존재하는 이유")
    order = {s: i for i, s in enumerate(VALID_SCOPES)}
    if order.get(rec["scope"], -1) < order.get(required_scope, 99):
        v.append(f"동의 범위 부족: {rec['scope']} < 요구 {required_scope}")
    if time.time() > rec["expires_at"]:
        v.append("동의 만료")
    return v

def clone_guard(clone_fn):
    """클로닝 함수를 동의 게이트로 감싼다 — 위반 시 PermissionError."""
    def guarded(text, ref_path, consent, required_scope="education_lab", **kw):
        v = validate_consent(consent, required_scope)
        if v:
            raise PermissionError(f"동의 계약 위반 — 클로닝 거부: {v}")
        return clone_fn(text, ref_path, **kw)
    return guarded

# ── 자가 점검 ──
ok = make_consent_record("spk_1", "self_voice", "education_lab", True)
assert validate_consent(ok, "education_lab") == []
bad_scope = make_consent_record("spk_1", "self_voice", "education_lab", True)
assert validate_consent(bad_scope, "production") == ["동의 범위 부족: education_lab < 요구 production"]
bad_src = make_consent_record("spk_1", "stolen_recording", "education_lab", True)
assert "미허용 출처" in validate_consent(bad_src, "education_lab")[0]

def _fake_clone(text, ref_path):
    return f"cloned({text}, {ref_path})"

guarded = clone_guard(_fake_clone)
assert guarded("안녕하세요", "/ref.wav", ok) == "cloned(안녕하세요, /ref.wav)"
try:
    guarded("안녕하세요", "/ref.wav", bad_scope, required_scope="production")
    raise SystemExit("게이트 방어 실패!")
except PermissionError as e:
    print("게이트 방어 확인 ✅ →", str(e)[:60], "…")
print("동의 계약 v1 + 클론 게이트 검증 통과 ✅")
print("⚠️ v2(4-EL)에서는 source_type에서 synthetic_machine이 빠지고 third_party_transfer·processor가 추가된다 (2.6에서).")


In [ ]:
# ▶ 데모 — 동의 게이트가 '무엇을 막는지' 눈으로 확인하기 (초보자용)
# 동의 범위가 부족하면 클로닝이 실행되지 않는다 — 이 '막음'이 권리의 핵심이다.

rec = make_consent_record("spk_1", "real_voice_consented", "education_lab",
                          notice_given=True, valid_days=30)
print("만든 동의 기록:", rec["source_type"], "/", rec["scope"])

viol = validate_consent(rec, required_scope="production")   # 좁은 범위로 넓은 것을 요구
print("production 요구 시 위반:", viol or "없음 (통과)")

# 클론 게이트가 실제로 막는지 — 위반 시 PermissionError
try:
    clone_guard(lambda t, r: "합성음")("합성할 문장", None, rec)   # 동의 통과 전 호출
    print("⚠️ 막히지 않음 (게이트 확인 필요)")
except PermissionError:
    print("✅ 게이트가 정상적으로 차단했다: 동의 없이 합성 불가")
print("데모 통과 ✅ — 권리는 기술이 아니라 '게이트'가 지킨다")


In [ ]:
# ═══ 2.1 이중 저울 — CER(내용) + SECS(목소리) + 3점 캘리브레이션 (4-V) ═══
# ▶ 이중 저울: cer(내용) + secs(목소리). 두 개를 따로 재야 한다.
#   _MockEncoder는 결정적 인코더(기본주파수·스펙트럼 중심·RMS·영교차율) — resemblyzer의 계약만 흉내.
#   ⚠️ secs는 절대 눈금이 아니다 — 3점 캘리브레이션(자기>같은>다른) 대비로 판독해야 한다.
import numpy as np

def cer(ref, hyp):
    """문자 오차율 — 공백 제거 후 Levenshtein / 참조 길이. 3-ST의 왕복 심판."""
    r, h = ref.replace(" ", ""), hyp.replace(" ", "")
    prev = list(range(len(h) + 1))
    for i, rc in enumerate(r, 1):
        cur = [i]
        for j, hc in enumerate(h, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (rc != hc)))
        prev = cur
    return prev[-1] / max(len(r), 1)

class _MockEncoder:
    """SECS 저울 시연용 결정적 인코더 — resemblyzer VoiceEncoder의 계약만 흉내.
    실습에서는: from resemblyzer import VoiceEncoder; enc = VoiceEncoder("cpu")"""
    def embed_utterance(self, wav):
        x = np.asarray(wav, dtype=np.float64)
        x = x - x.mean()
        n = len(x)
        if n < 8:
            return np.zeros(4, dtype=np.float64)
        win = x * np.hanning(n)
        fft = np.abs(np.fft.rfft(win))
        freqs = np.fft.rfftfreq(n, d=1 / 16000)
        dom = float(freqs[np.argmax(fft)]) if fft.max() > 1e-9 else 0.0
        centroid = float(np.sum(freqs * fft) / max(fft.sum(), 1e-9))
        rms = float(np.sqrt((x ** 2).mean()))
        zc = float(np.mean(np.abs(np.diff(np.sign(x))) > 0)) if n > 1 else 0.0
        v = np.array([np.log10(rms + 1e-12), dom / 1000.0, centroid / 1000.0, zc])
        nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else v

enc = _MockEncoder()

def secs(wav_a, wav_b):
    """화자 임베딩 코사인 유사도 — 0~1. ⚠️ 절대 눈금 아님: 3점 기준선 대비 상대 판독."""
    ea, eb = enc.embed_utterance(wav_a), enc.embed_utterance(wav_b)
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

def _tone(freq, dur=1.0, seed=42):
    rng = np.random.default_rng(seed)
    t = np.arange(int(16000 * dur)) / 16000
    return (0.5 * np.sin(2 * np.pi * freq * t) + 0.05 * rng.normal(size=t.size)).astype(np.float32)

# ── 자가 점검 ──
assert abs(cer("환불 처리 도와드리겠습니다", "환불 처리 도와드리겠습니다")) < 1e-9
assert cer("안녕하세요", "안녕") > 0.3     # 일부 누락 → 오차 존재
assert abs(cer("완전히 다른 문장입니다", "반대의 내용이지요")) > 0.5

w_ref = _tone(200)          # 화자 A — 우세 주파수 200Hz
w_other = _tone(200, seed=7)   # 같은 화자·다른 발화 (잡음만 다름)
w_en = _tone(330)           # 다른 화자 — 우세 주파수 330Hz

CAL = {"자기 자신": secs(w_ref, w_ref),
       "같은 화자·다른 발화": secs(w_ref, w_other),
       "다른 화자": secs(w_ref, w_en)}
for k, v in CAL.items():
    print(f"  {k:14s}: {v:.3f}")
assert CAL["자기 자신"] > 0.99
assert CAL["같은 화자·다른 발화"] > CAL["다른 화자"], "기준선 격차 실패"
print("이중 저울(CER·SECS) + 3점 캘리브레이션 검증 통과 ✅")
print("⚠️ 실제 resemblyzer도 '기준선 대비 상대 판독'이 규칙 — 절대 임계값이 아니다.")


In [ ]:
# ═══ 2.2 A/B 삼종(파형 정합) + 임베딩 계약(validate_emb) (4-Q · 4-E) ═══
# ▶ A/B 삼종은 '두 오디오가 같은 신호인가'를 파형으로 검증: 길이비·RMS(dB)차·상관계수.
#   validate_emb: 화자 임베딩이 계약({dim, norm:1.0, dtype:float32})을 지키는지 — 단위벡터면 코사인=내적.
import numpy as np

def ab_triple(x, y, sr=16000):
    """A/B 삼종 — 대조군 A vs 변형 B의 파형 정합. 완전 동일 = (1.0, 0.0, 1.0)."""
    dur_ratio = len(x) / max(len(y), 1)
    rms = lambda v: 20 * np.log10(np.sqrt((v ** 2).mean()) + 1e-12)
    n = min(len(x), len(y))
    corr = float(np.corrcoef(x[:n], y[:n])[0, 1]) if n > 1 else float("nan")
    return {"dur_ratio": dur_ratio, "rms_db_diff": rms(x) - rms(y), "corr": corr}

def validate_emb(e):
    """임베딩 계약 검사 — 위반 목록 반환. (dim · L2=1 · float32)"""
    v = []
    if e.shape != (EMB_CONTRACT["dim"],):
        v.append(f"차원 {e.shape} ≠ ({EMB_CONTRACT['dim']},)")
    if abs(float(np.linalg.norm(e)) - EMB_CONTRACT["norm"]) > 1e-4:
        v.append(f"노름 {np.linalg.norm(e):.4f} ≠ 1 — 구면 이탈")
    if str(e.dtype) != EMB_CONTRACT["dtype"]:
        v.append(f"dtype {e.dtype}")
    return v

# ── 자가 점검 ──
EMB_CONTRACT = {"dim": 4, "norm": 1.0, "dtype": "float32"}
good = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)
assert validate_emb(good) == []
bad_dim = np.zeros(3, dtype=np.float32)
assert "차원" in validate_emb(bad_dim)[0]
bad_norm = np.array([2.0, 0.0, 0.0, 0.0], dtype=np.float32)
assert "노름" in validate_emb(bad_norm)[0]
print("임베딩 계약 검증 통과 ✅ (단위벡터 계약이므로 코사인 = 내적)")

x = _tone(200, dur=0.5)
y = _tone(200, dur=0.5, seed=99)
t = ab_triple(x, y)
print("A/B 삼종(동일 주파수·다른 잡음):", {k: round(v, 3) for k, v in t.items()})
assert abs(t["dur_ratio"] - 1.0) < 1e-6 and abs(t["rms_db_diff"]) < 3.0 and t["corr"] > 0.95
print("A/B 삼종 검증 통과 ✅")


In [ ]:
# ═══ 2.3 화자 등록부(SpeakerRegistry) + 정합성 감사(audit_registry) (4-M) ═══
# ▶ SpeakerRegistry.register: 전사 수==오디오 수·발화 2개 이상·동의·파일 존재를 한 번에 검사.
#   audit_registry: cps(초당 글자)가 상식 범위 2.0~10.0을 벗어나면 의심 — '너무 빠른/느린' 전사 감사.
import os, tempfile, wave

def cps_of(text, dur):
    """발화 속도 — 공백 제거 글자 수 / 초. 전사 정합성의 상식 범위 2.0~10.0."""
    return len(text.replace(" ", "")) / max(dur, 1e-6)

def _wav_dur(path):
    """WAV 지속시간(초) — stdlib만 사용 (실습은 librosa.load)."""
    with wave.open(path, "rb") as w:
        return w.getnframes() / float(w.getframerate())

class SpeakerRegistry:
    """화자별 동의를 강제하는 등록부 — 등록: 전사수==오디오수, 발화≥2, 동의 검증, 파일 존재."""
    def __init__(self):
        self._d = {}

    def register(self, speaker_id, ref_paths, ref_texts, consent):
        assert len(ref_paths) == len(ref_texts), \
            f'{speaker_id}: 오디오 {len(ref_paths)}개 vs 전사 {len(ref_texts)}개 불일치'
        assert len(ref_paths) >= 2, \
            f'{speaker_id}: 발화가 {len(ref_paths)}개 — intra 기준선을 만들려면 최소 2개 필요'
        assert validate_consent(consent, "education_lab") == [], \
            f'{speaker_id}: 동의 계약 위반'
        for p in ref_paths:
            assert os.path.exists(p), f'{speaker_id}: 파일 없음 {p}'
        self._d[speaker_id] = {"paths": list(ref_paths), "texts": list(ref_texts),
                               "consent": consent}
        return self

    def ids(self):
        return sorted(self._d.keys())

    def get(self, sid):
        return self._d[sid]

    def guard(self, sid):
        """해당 화자에 대해서만 클로닝을 허가한다."""
        if sid not in self._d:
            raise PermissionError(f"미등록 화자: {sid} — 복제할 수 없습니다.")
        return clone_guard(self._d[sid]["consent"])

    def __len__(self):
        return len(self._d)

def audit_registry(reg):
    """전사와 오디오의 정합성을 발화 속도로 교차 검증 — 에러 없이 품질만 망가는 '전사 교차 오염'을 잡는다."""
    problems = []
    for sid in reg.ids():
        r = reg.get(sid)
        for i, (p, t) in enumerate(zip(r["paths"], r["texts"]), start=1):
            dur = _wav_dur(p)
            cps = cps_of(t, dur)
            if not (2.0 <= cps <= 10.0):
                problems.append(f"{sid} utt{i}: {cps:.1f} 자/초 (상식 범위 2~10 밖)")
    return problems

# ── 자가 점검: 1초(16000프레임)짜리 WAV 임시 생성 ──
def _make_wav(path, dur=1.0):
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000)
        w.writeframes((np.zeros(int(16000 * dur), dtype=np.int16)).tobytes())

tmp = tempfile.mkdtemp()
p1, p2 = os.path.join(tmp, "s1_a.wav"), os.path.join(tmp, "s1_b.wav")
_make_wav(p1, 1.0); _make_wav(p2, 1.0)

consent_ok = make_consent_record("spk_1", "self_voice", "education_lab", True)
reg = SpeakerRegistry().register("spk_1", [p1, p2],
                                 ["안녕하세요 고객님", "네 말씀하세요"], consent_ok)
assert len(reg) == 1 and reg.guard("spk_1") is not None
try:
    reg.guard("spk_2"); raise SystemExit("미등록 화자 방어 실패!")
except PermissionError:
    pass

probs = audit_registry(reg)
print("정합성 감사:", "통과 ✅" if not probs else f"의심 {probs}")
assert probs == []
print("등록부 + 정합성 감사 검증 통과 ✅ (2~10 자/초 상식 범위)")


In [ ]:
# ═══ 2.4 토큰 상한 — 생성 스텝을 수식으로 예측 (4-O · 4-Q) ═══
# ▶ 토큰 상한을 수식으로 예측: mnt_for(t) = max(8, ceil(t*3.0)*1.5) → est_max_new_tokens → predicted_steps.
#   '얼마나 많은 스텝이 생성될지 미리 안다' = 지연·원가를 미리 안다 (4-O/4-Q).
CODEC_HZ = 12.0          # 모델명 "12Hz" — 초당 코덱 프레임 수
CHARS_PER_SEC = 3.0      # 보수적(느린) 발화 가정
SAFETY = 1.5             # 여유 배수
FLOOR = 96               # 짧은 문장의 최소 여유
NUM_CODE_GROUPS = 8      # probe()로 실측한 코드그룹 수 (문서값 32가 아닌 실제)

def mnt_for(text, hz=CODEC_HZ, cps=CHARS_PER_SEC, safety=SAFETY, floor=FLOOR):
    """max_new_tokens 상한 — 4-M의 공식. 과잉 할당이 지연을 만든다."""
    n = len(str(text).replace(" ", ""))
    return int(max(max(n / cps, 1.0) * hz * safety, floor))

def est_max_new_tokens(text, codec_hz=CODEC_HZ):
    """4-Q의 변형 — 발화 초 추정치에서 토큰 상한을 구한다. (mnt_for와 같은 골격)"""
    n = len(str(text).replace(" ", ""))
    est_sec = max(n / CHARS_PER_SEC, 1.0)
    return int(max(est_sec * codec_hz * SAFETY, FLOOR))

def predicted_steps(audio_sec, hz=CODEC_HZ, groups=NUM_CODE_GROUPS):
    """오디오 n초 → (코덱 프레임 수, 예측 생성 스텝 수) — 4-O의 병목 해부."""
    frames = audio_sec * hz
    return int(frames), int(frames * groups)

# ── 자가 점검 ──
t = "네 고객님, 주문번호 확인해 드리겠습니다. 잠시만 기다려 주세요."   # 4-M TEST_TEXT 급
assert mnt_for(t) == est_max_new_tokens(t)          # 같은 골격 → 같은 결과
assert mnt_for("안녕") == FLOOR                       # 짧은 문장은 floor에 걸린다
f, s = predicted_steps(2.0)
assert f == 24 and s == 192                          # 2초 → 24프레임 × 8그룹 = 192스텝
print(f"토큰 상한 검증 통과 ✅ — 예: '{t[:16]}…' → max_new_tokens {mnt_for(t)}")
print(f"predicted_steps(2.0s) = 프레임 {f} × 그룹 8 = {s}스텝 — 소스에서 구조를 읽는다(4-O)")


In [ ]:
# ═══ 2.5 최적화 도구 — probe(구조 읽기) · is_pareto · 운영점 계약 (4-O) ═══
# ▶ probe: 소스 코드(함수 시그니처)를 읽어 구조를 파악 — '추측하지 말고 소스를 보라'.
#   is_pareto: 어느 축으로도 못 이기는 점 제거 → synth_optimized가 파레토 중 운영점을 골라 계약화.
import numpy as np

def probe(obj, *path, default=None):
    """설치된 객체의 실제 구조를 getattr로 읽는다 — 문서값을 믿지 않는다 (4-O 셀 1)."""
    cur = obj
    for p in path:
        cur = getattr(cur, p, None)
        if cur is None:
            return default
    return cur

def is_pareto(r, others):
    """더 빠르면서(sec 작고) 더 닮은(secs 큰) 점이 존재하면 열등 — 4-O 파레토 프론티어."""
    return not any((o["sec"] <= r["sec"] and o["secs"] >= r["secs"])
                   and (o["sec"] < r["sec"] or o["secs"] > r["secs"])
                   for o in others if o is not r)

# 4-O의 INFER_CONTRACT — 운영점을 계약으로 고정해 다음 세션이 물려받게 한다
INFER_CONTRACT = {
    "model": "qwen3-tts-12hz-0.6b-base",
    "generate_kwargs": {"max_new_tokens": None,   # synth_optimized가 문장 길이로 채운다
                        "top_p": 0.9, "temperature": 0.9},
    "batch_size": 4, "note": "4-O 파레토에서 선택된 운영점 (배치=핵심 레버)",
}

def synth_optimized(texts, engine):
    """운영점으로 고정 호출 — max_new_tokens를 문장들 중 최대로 넣어 배치 합성.
    engine: generate_voice_clone(texts, **kwargs) → (wavs, sr) 계약을 가진 mock/실엔진."""
    texts = [texts] if isinstance(texts, str) else list(texts)
    gk = dict(INFER_CONTRACT["generate_kwargs"])
    gk["max_new_tokens"] = max(mnt_for(t) for t in texts)
    wavs, sr = engine.generate_voice_clone(text=texts, **gk)
    return [{"text": t, "audio": np.asarray(w, dtype=np.float32), "sr": sr,
             "engine": engine.name} for t, w in zip(texts, wavs)]

class _MockEngine:
    """배치 합성 계약만 흉내 내는 mock — wavs는 (배치, n) ndarray."""
    name = "mock/qwen3-tts"
    def __init__(self, sr=16000, dur=0.8):
        self.sr, self.dur = sr, dur
    def generate_voice_clone(self, text, **kwargs):
        n = len(text) if isinstance(text, list) else 1
        t = np.arange(int(self.sr * self.dur)) / self.sr
        w = 0.2 * np.sin(2 * np.pi * 180 * t)
        return [w.astype(np.float32) for _ in range(n)], self.sr

# ── 자가 점검 ──
class _Cfg:
    talker_config = type("T", (), {"num_code_groups": 8, "num_frames_per_second": 12})()

assert probe(_Cfg, "talker_config", "num_code_groups") == 8
assert probe(_Cfg, "talker_config", "없는_속성", default=0) == 0

pts = [{"sec": 1.0, "secs": 0.98}, {"sec": 0.7, "secs": 0.95},   # 파레토 (느리지만 닮음 / 빠르고 약간 덜 닮음)
       {"sec": 0.9, "secs": 0.91}]                               # 열등: 0.7s·0.95가 지배
pareto = [p for p in pts if is_pareto(p, pts)]
assert len(pareto) == 2 and pts[2] not in pareto
print("probe·is_pareto 검증 통과 ✅ — 파레토 남은 점:", len(pareto))

outs = synth_optimized(["네 안녕하세요", "환불 도와드리겠습니다"], _MockEngine())
assert len(outs) == 2 and all(o["sr"] == 16000 for o in outs)
assert all(o["engine"] == "mock/qwen3-tts" for o in outs)
print("synth_optimized(운영점 고정) 검증 통과 ✅ — 배치 크기", len(outs))
print("4-O 결론: 배치(레버 C)가 시간을 지우는 구조적 축 — 길이 버킷팅으로 더 이득.")


In [ ]:
# ═══ 2.6 ElevenLabs 감정 클로닝 어댑터 (4-EL) — 정의만, API 미호출 ═══
# ▶ ElevenLabs 어댑터: 동의 계약 v2(제3자 반출·processor 추가)·감정 태그 5종·원가 계량기(CostMeter).
#   API 호출은 하지 않음 — 정의만. '감정 태그도 문자다': 표현력과 원가가 같은 축에 있다.
# 유료 API — 이 셀은 로직(원가·감정 태그)만 정의한다. 실제 호출은 실행하지 않는다.
# 실습: pip install elevenlabs; client = ElevenLabs(api_key=...)  (키는 환경변수/파일에만)

# 감정 태그 5종 — 오디오 태그는 v3 전용 (v2 계열은 태그를 '읽어버린다')
EMOTIONS = {
    "중립":   "",
    "사과":   "[apologetic] ",
    "긴급":   "[urgent] ",
    "따뜻함": "[warm] ",
    "속삭임": "[whispers] ",
}

class CostMeter:
    """원가 계량기 — 문자당 과금이라는 물리 법칙. 태그 문자도 비용이다."""
    def __init__(self):
        self.calls, self.chars, self.log = 0, 0, []
    def charge(self, text, tag=""):
        n = len(text)                       # ElevenLabs 과금 단위 = 문자 수
        self.calls += 1; self.chars += n
        self.log.append((tag, n))
        return n
    def report(self):
        print(f"누적 호출 {self.calls}회 | 과금 문자 {self.chars:,}자")
        for tag, n in self.log:
            print(f"   {tag:22s} {n:>5,}자")

BUDGET_CHARS = 3000                         # 수업용 상한 — 초과 시 즉사시켜 사고 과금을 막는다
def budget_guard(text, tag):
    n = METER.charge(text, tag)
    assert METER.chars <= BUDGET_CHARS, \
        f"예산 초과: 누적 {METER.chars:,}자 > 상한 {BUDGET_CHARS:,}자 — 무한 루프/오타 과금 방지 게이트"
    return n

# v2 동의 계약 — API 실습이 새 조항(제3자 반출)을 강제했다
CONSENT_KEYS_V2 = CONSENT_CONTRACT_KEYS + ("third_party_transfer", "processor")
VALID_SOURCE_TYPES_V2 = ("real_voice_consented", "self_voice")   # ⚠️ synthetic_machine 제외: 회색지대

def tts(text, tag, voice_id, model_id="eleven_multilingual_v2", voice_settings=None):
    """ElevenLabs 호출 — 이 노트에서는 정의만 (네트워크 미호출).
    실습: budget_guard(text, tag) → client.text_to_speech.convert(...) → mp3 → 16k wav"""
    raise NotImplementedError("API 호출은 실습 노트북에서만 (키 필요)")

# ── 자가 점검 ──
METER = CostMeter()
demo = "[excited] 안녕하세요!"
assert budget_guard(demo, "demo") == len(demo)          # 태그 문자도 과금된다
assert METER.chars == len(demo)
big = "[excited] " + "안녕" * BUDGET_CHARS              # 초과 유도
try:
    budget_guard(big, "big"); raise SystemExit("원가 게이트 방어 실패!")
except AssertionError as e:
    print("원가 게이트 방어 확인 ✅ →", str(e)[:55], "…")
print("감정 태그·원가 계량기 검증 통과 ✅")
print("주목: 감정 태그도 '문자'다 — 표현력과 원가가 같은 축에 있다 (4-EL 셀 9).")
print("⚠️ v2 source_type: real/self만 허용 — 타 서비스 출력물 업로드는 회색지대.")


## 2.7 📄 요약 — 계약 진화표 · 카드 2장

### 동의 계약 v1 → v2 (4-V → 4-EL)

| 키 | v1 (4-V) | v2 (4-EL) | 의미 |
|---|---|---|---|
| `source_type` | 3종 (real/synthetic/self) | 2종 (real/self) | **synthetic_machine 제외** — 타 서비스 출력물 업로드는 회색지대 |
| `third_party_transfer` | — | 추가 | 제3자 반출 동의 — **API 업로드 = 반출** |
| `processor` | — | 추가 | 처리자 명시 (누가 데이터를 본다) |
| `scope` | 3단계 | 3단계 | education_lab < internal_poc < production |
| 유효기간 | 30일 | 30일 | 만료 시 재동의 |

> 교훈: 계약은 고정된 형태가 아니라 **실습의 경계가 바뀔 때 진화**한다. 외부 API가 '반출' 조항을 만들어냈다.

### 📄 임베딩 카드 (4-E)

```
계약: (dim,) · L2=1(구면) · float32 — 코사인 = 내적
기전: 1.6초 창으로 훑기 → 부분 임베딩 평균 → 재정규화
길이: 벡터는 일찍 안정된다 — 그 이름이 '3초의 무게'
잡음: 5dB에서 벡터는 이미 크게 밀려난다 — 4-V 셀 8의 기전
지도: 내>간 코사인 — 클로닝의 전제 = diarization의 원리
```

### 📄 클로닝 카드 1호 (4-V)

```
기전:   임베딩형 vs 프롬프트 연속형 — 입력 계약(전사 필요 여부)이 갈림길
저울:   SECS (3점 캘리브레이션 대비 상대 판독) + CER — 목소리와 내용은 다른 저울
실험:   3초 vs 10초(격차의 '작음'이 위험의 크기) · 5dB 오염(두 저울의 상이한 붕괴)
계약:   동의 계약(출처/범위/고지/만료) + clone_guard — 동의는 프롬프트가 아니라 게이트
양쪽 문: 입력 게이트 + 출력 워터마크 서명 — 둘 중 하나만으로는 반쪽
다음:   다중 엔진 클로닝 비교 (Qwen3-TTS / Fish / MOSS) — 같은 두 저울로
```


## 2.8 [REAL] 실물 실행 — ElevenLabs 감정 클로닝 + 로컬 화자 임베딩

> 2.6에서 `NotImplementedError` 였던 `tts()` 를 **실제 API**로 구현합니다 (키 있을 때만).
> 준비: `.env` 에 `ELEVENLABS_API_KEY` 등록 → 커널 재시작. `bash setup_apple_silicon.sh cloning`
> 로컬 대안으로 `resemblyzer` 화자 임베딩(2.1 SECS 재료)도 실측합니다.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if ae.has("elevenlabs") and ae.key_present("ELEVENLABS"):
    from elevenlabs.client import ElevenLabs
    client = ElevenLabs(api_key=ae.KEYS["ELEVENLABS"])

    # 클로닝용 음성 샘플 — 없으면 MeloTTS 합성 데모 샘플 생성 (실제 클로닝엔 실제 동의 음성 권장)
    sample = ae.HERE / "assets" / "speaker_sample.wav"
    if not sample.exists():
        if ae.has("melotts"):
            from melo.api import TTS as MeloTTS
            m = MeloTTS(language="KR", device="cpu")
            m.tts_to_file("안녕하세요. 이 목소리로 상담 응대를 부탁드립니다.",
                          m.hps.data.spk2id["KR"], str(sample))
        else:
            print("⚠️ 샘플 없음: assets/speaker_sample.wav (10~30초 동의 음성) 을 직접 준비하세요")
    if sample.exists():
        with open(sample, "rb") as f:
            voice = client.voices.clone(
                name="my-lab-demo",
                files=[(sample.name, f.read())],
                description="AICC my-lab 데모용 목소리")
        text = "[warm] " + "네 고객님, 문의하신 내용 확인해 드리겠습니다."
        budget_guard(text, "eleven_clone")                       # 원가 계량기 — 태그 문자도 과금
        audio = client.text_to_speech.convert(
            text=text, voice_id=voice.voice_id,
            model_id="eleven_multilingual_v2", output_format="mp3_44100_128")
        out = ae.HERE / "assets" / "clone_out.mp3"
        out.write_bytes(b"".join(audio))
        print("클로닝 응답 저장:", out, f"({out.stat().st_size:,} B)")
        METER.report()
        print("ElevenLabs 감정 클로닝 왕복 통과 ✅ ([warm] 태그 포함)")
else:
    print("ELEVENLABS_API_KEY 없음 → 클로닝 셀 스킵 (.env 등록 후 커널 재시작)")


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not ae.has("resemblyzer"):
    print("resemblyzer 미설치 → 스킵.  bash setup_apple_silicon.sh cloning")
else:
    import numpy as np
    from resemblyzer import VoiceEncoder, preprocess_wav
    enc = VoiceEncoder()

    wav = ae.ensure_test_audio()
    audio, sr = ae.load_wav(wav)
    # '다른 화자' 시뮬레이션: 다른 주파수 보강 — 실제론 서로 다른 녹음을 넣으면 됨
    t = np.arange(audio.size) / sr
    other = (0.5 * np.sin(2 * np.pi * 260 * t) + 0.5 * audio).astype(np.float32)
    other_path = ae.save_wav(ae.HERE / "assets" / "_other_speaker.wav", other, 16000)

    e1 = enc.embed_utterance(preprocess_wav(str(wav), 16000))
    e2 = enc.embed_utterance(preprocess_wav(str(other_path), 16000))
    same = float(np.dot(e1, e1) / (np.linalg.norm(e1) ** 2))
    sim = float(np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2)))
    print(f"화자 임베딩 — 자기 유사도 {same:.3f} | 변조 화자 유사도 {sim:.3f}")
    assert sim < same, "다른 화자는 유사도가 낮아야 한다"
    print("resemblyzer 실물 임베딩 통과 ✅ (2.1 SECS 이중 저울의 실물 재료)")


# 3. 실험 진행 방법 🧪

## 3-1. 클로닝 7단계 워크플로우 (6개 노트북 공통)

```
① 동의 계약   make_consent_record → validate_consent → clone_guard 장착
② 참조 수집   마이크 녹음 / 파일 업로드 / (4-EL: 라이브러리 보이스)
③ 참조 전사   Whisper(4-M) 또는 수기(4-V/4-Q) — ICL 모드의 생명줄
④ 참조 게이트 발화 속도·길이·포맷 상식 검사 (입력측 문)
⑤ 클로닝      게이트 통과 참조만 엔진에 (4-V _clone / 4-Q generate_voice_clone)
⑥ 이중 심판   dual_scale: 내용 CER × 목소리 SECS (3점 기준선 대비)
⑦ 정량 비교   A/B 삼종 · intra/inter · 파레토 → 운영점 계약화
```

- 4-M 셀 3b의 강조: **검청(귀로 확인)이 가설을 사실로 승격시키는 유일한 방법** — ASR 전사는 증거가 아니라 가설이다.
- 4-Q 셀 10의 강조: **클로닝 호출 직전에 반드시 `clone_guard(CONSENT)`** — 게이트는 등록 시점이 아니라 **호출 시점**에 다시 확인한다.

## 3-2. 6개 노트북별 가이드 + macOS 실습 치환표

| 노트북 | 핵심 실험 | 측정 | macOS 치환 |
|---|---|---|---|
| 4-V | 3초 vs 10초 참조 · 5dB 오염 · 유사도 지도 | CER×SECS | gTTS 대역 + resemblyzer 로컬 |
| 4-E | 부분 발화 해부 · 안정화 곡선 · SNR 스윕 | 임베딩 cos | resemblyzer 로컬 ✅ |
| 4-M | 등록부 · 교차 행렬 · 임계값 · 전사 오염 | intra/inter/GAP | resemblyzer+Whisper 로컬, 모델만 Colab |
| 4-O | 병목 해부 · 레버 A~E · 파레토 | sec/RTF × secs | 모델 실습만 Colab |
| 4-Q | ICL vs x-vector · ref_text 제거 · A/B 삼종 | SECS+A/B 삼종 | 모델 실습만 Colab |
| 4-EL | 감정 태그 5종 · 태그 vs 슬라이더 | SECS+스타일 좌표+원가 | API 키만 있으면 로컬 ✅ |

**Colab ↔ 로컬 연결법**: 모델 실습의 산출물(16k wav + `make_tts_record` 메타데이터)을 파일로 내려받아 로컬에서 SECS·CER·정합성 감사로 재심사 — 저울은 로컬, 엔진은 클라우드.

## 3-3. 판단 기준 모음

- **SECS**: 절대 임계값이 아니다 — 3점 캘리브레이션(자기=1.0, 같은 화자>다른 화자) 사이 어디인가로 판독.
- **intra/inter**: `GAP < 0.10` → 해상도 낮음, `GAP <= 0` → 화자 구별 불가 (데이터부터 의심).
- **A/B 삼종**: 완전 동일 = `(1.0, 0.0, 1.0)` — dur_ratio·rms_db_diff·corr를 함께 봐야 하나의 그림.
- **파레토**: "더 빠르고 더 닮은 점이 있으면 열등" — 운영점은 파레토 프론티어에서 요구사항으로 선택.
- **의도적 위반** 3종: 미허용 출처 · 범위 부족 · ICL에서 ref_text 누락 — **모두 예외 타입이 아니라 증상 패턴으로 검증** (가드가 실제로 막는지).


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. 클로닝 파이프라인 — 등록부 중심

```
               ┌──────────────────────────────────────────────┐
               │  SpeakerRegistry (화자 단위 동의 강제)          │
               │  register: 전사수==오디오수, 발화≥2, 파일 존재    │
               │  guard(sid): 해당 화자만 clone_guard 통과       │
               └───────────────┬──────────────────────────────┘
                               │ 참조(경로·전사·동의)
   [동의 계약 게이트] ──→ 참조 오디오 ──→ [참조 전사/발화속도 게이트] ──→ 엔진
       (clone_guard)             │            (cps 2~10 · 3초)          (Qwen3-TTS)
                                 └───────────────┬──────────────────────┘
                                                ↓ 합성 (16k 계약)
   출력측 문: [워터마크 서명] ←── 출력 ──→ [이중 저울] CER(내용) × SECS(목소리)
```

- **입력에서 권리를 지키고(동의 게이트), 출력에서 고지를 자동화한다(워터마크 서명)** — 둘 중 하나만으로는 반쪽 (4-V 셀 10).

## 4-2. 계약 3층 구조 (클로닝 파트에서 상속/추가된 것)

| 층 | 계약 | 검증 함수 | 역할 |
|---|---|---|---|
| 1. 동의 | `CONSENT_CONTRACT_KEYS` (+v2: 반출/처리자) | `validate_consent` | 권리 — 클로닝 이전에 통과 |
| 2. 임베딩 | `EMB_CONTRACT {dim, norm:1, dtype}` | `validate_emb` | 저울의 무결성 — 코사인=내적 보장 |
| 3. TTS | `TTS_CONTRACT_KEYS {audio, sr, ...}` (4-Q) | `validate_tts_contract` | 산출물 규격 — sr 16k 강제 |

## 4-3. 최적화 계약화 — 실험 결과가 다음 세션에 전달되는 형태

```
레버 실험(4-O) → 파레토 프론티어 → 요구사항 REQ → 운영점 자동 선택
     → INFER_CONTRACT (배치 크기·샘플링·모델) → synth_optimized(계약대로 호출)
     → 4-Q가 물려받아 재사용
```
- "선택을 계약으로 고정한다" — 휴리스틱이 아니라 **다음 세션이 물려받을 형태**로 남긴다.

## 4-4. macOS(Apple Silicon) 배포 아키텍처

```
[모델 실습: Colab T4]  Qwen3-TTS-0.6B-Base ──→ 16k wav + 메타데이터(JSON)
                        │ (4-M/4-O/4-Q)
[로컬 Jupyter: Apple Silicon]  resemblyzer(SECS·임베딩) · gTTS(대역) · 계약·감사 · ElevenLabs API
```

## 4-5. 설계 원칙 (6개 노트북이 공유하는 5줄)

1. **게이트는 프롬프트가 아니라 코드다** — 동의는 '잘 부탁'이 아니라 `clone_guard`로 통과 여부를 강제한다.
2. **저울부터 검증한다** — 쓰기 전에 3점 캘리브레이션. SECS는 절대 눈금이 아니다.
3. **저울과 재료를 분리해 읽는다** — 5dB 오염에서 SECS 하락은 엔진 실패 전에 참조 벡터의 이동이다.
4. **문서가 아니라 소스·실측** — `probe()`로 구조를 읽고, `_validate_languages`처럼 모델에게 직접 묻는다.
5. **의도적 위반을 반복한다** — 게이트가 실제로 막는지, ref_text를 빼면 진짜 실패하는지 증상으로 확인.
